In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the original dataset
df = pd.read_csv(
    r"C:\Users\sandr\Downloads\luminar\ncr_ride_bookings.csv"
)

# Create a separate DataFrame for cleaning
df_clean = df.copy()

In [3]:
# How many exact duplicate rows are present?
print("Duplicate rows:", df_clean.duplicated().sum())

# Remove exact duplicate rows
df_clean = df_clean.drop_duplicates()

# What is the dataset shape after removing duplicates?
print("Shape after removing duplicates:", df_clean.shape)

Duplicate rows: 0
Shape after removing duplicates: (150000, 21)


In [4]:
# Convert Date into datetime format
df_clean["Date"] = pd.to_datetime(
    df_clean["Date"], errors="coerce"
)

# Check the Date data type
print("Date data type:", df_clean["Date"].dtype)

# Check for missing or invalid dates
print("Missing or invalid dates:", df_clean["Date"].isna().sum())

Date data type: datetime64[us]
Missing or invalid dates: 0


In [5]:
# What are the first 10 time values?
print(df_clean["Time"].head(10))

# What is the current data type of Time?
print("Time data type:", df_clean["Time"].dtype)

# Are any time values missing?
print("Missing time values:", df_clean["Time"].isnull().sum())

0    12:29:38
1    18:01:39
2    08:56:10
3    17:17:25
4    22:08:00
5    09:44:56
6    15:45:58
7    17:37:37
8    12:49:09
9    19:06:48
Name: Time, dtype: str
Time data type: str
Missing time values: 0


In [6]:
# Convert Time into datetime format
df_clean["Time"] = pd.to_datetime(
    df_clean["Time"],
    format="%H:%M:%S",
    errors="coerce"
).dt.time

# Check the converted time values
print(df_clean["Time"].head(10))

# Check the data type
print("Time data type:", df_clean["Time"].dtype)

# Check for missing or invalid times
print("Missing or invalid times:", df_clean["Time"].isnull().sum())

0    12:29:38
1    18:01:39
2    08:56:10
3    17:17:25
4    22:08:00
5    09:44:56
6    15:45:58
7    17:37:37
8    12:49:09
9    19:06:48
Name: Time, dtype: object
Time data type: object
Missing or invalid times: 0


In [7]:
# Remove extra spaces from text columns
for col in df_clean.select_dtypes(include=["object", "str"]).columns:
    df_clean[col] = df_clean[col].apply(
        lambda x: x.strip() if isinstance(x, str) else x
    )

In [8]:
# Convert numeric columns into numeric data types
numeric_columns = [
    "Avg VTAT",
    "Avg CTAT",
    "Booking Value",
    "Ride Distance",
    "Driver Ratings",
    "Customer Rating"
]

for col in numeric_columns:
    df_clean[col] = pd.to_numeric(
        df_clean[col], errors="coerce"
    )

In [9]:

# Convert event indicator columns into integers
indicator_columns = [
    "Cancelled Rides by Customer",
    "Cancelled Rides by Driver",
    "Incomplete Rides"
]

for col in indicator_columns:
    df_clean[col] = pd.to_numeric(
        df_clean[col], errors="coerce"
    ).fillna(0).astype("int64")



In [27]:


# Verify indicator columns after cleaning
print("\nMissing indicator values after cleaning:")
print(df_clean[indicator_columns].isnull().sum())

print("\nIndicator column data types:")
print(df_clean[indicator_columns].dtypes)


Missing indicator values after cleaning:
Cancelled Rides by Customer    0
Cancelled Rides by Driver      0
Incomplete Rides               0
dtype: int64

Indicator column data types:
Cancelled Rides by Customer    int64
Cancelled Rides by Driver      int64
Incomplete Rides               int64
dtype: object


In [12]:
# Check missing Avg VTAT values by booking status
print(pd.crosstab(
    df_clean["Booking Status"],
    df_clean["Avg VTAT"].isnull()
))

Avg VTAT               False  True 
Booking Status                     
Cancelled by Customer  10500      0
Cancelled by Driver    27000      0
Completed              93000      0
Incomplete              9000      0
No Driver Found            0  10500


In [13]:
# Check missing Avg CTAT values by booking status
print(pd.crosstab(
    df_clean["Booking Status"],
    df_clean["Avg CTAT"].isnull()
))

Avg CTAT               False  True 
Booking Status                     
Cancelled by Customer      0  10500
Cancelled by Driver        0  27000
Completed              93000      0
Incomplete              9000      0
No Driver Found            0  10500


In [14]:
# Check missing Booking Value by booking status
print(pd.crosstab(
    df_clean["Booking Status"],
    df_clean["Booking Value"].isnull()
))

Booking Value          False  True 
Booking Status                     
Cancelled by Customer      0  10500
Cancelled by Driver        0  27000
Completed              93000      0
Incomplete              9000      0
No Driver Found            0  10500


In [15]:
# Check missing Ride Distance values by booking status
print(pd.crosstab(
    df_clean["Booking Status"],
    df_clean["Ride Distance"].isnull()
))

Ride Distance          False  True 
Booking Status                     
Cancelled by Customer      0  10500
Cancelled by Driver        0  27000
Completed              93000      0
Incomplete              9000      0
No Driver Found            0  10500


In [16]:
# Check missing Payment Method values by booking status
print(pd.crosstab(
    df_clean["Booking Status"],
    df_clean["Payment Method"].isnull()
))

Payment Method         False  True 
Booking Status                     
Cancelled by Customer      0  10500
Cancelled by Driver        0  27000
Completed              93000      0
Incomplete              9000      0
No Driver Found            0  10500


In [17]:
# Check missing Driver Ratings by booking status
print(pd.crosstab(
    df_clean["Booking Status"],
    df_clean["Driver Ratings"].isnull()
))

Driver Ratings         False  True 
Booking Status                     
Cancelled by Customer      0  10500
Cancelled by Driver        0  27000
Completed              93000      0
Incomplete                 0   9000
No Driver Found            0  10500


In [31]:
# Check missing Customer Rating values by booking status
print(pd.crosstab(
    df_clean["Booking Status"],
    df_clean["Customer Rating"].isnull()
))

Customer Rating        False  True 
Booking Status                     
Cancelled by Customer      0  10500
Cancelled by Driver        0  27000
Completed              93000      0
Incomplete                 0   9000
No Driver Found            0  10500


### Why I Filled Only 3 Columns with 0

I filled missing values with `0` in **Cancelled Rides by Customer, Cancelled Rides by Driver, and Incomplete Rides** because these columns indicate whether an event occurred. I treated missing values as `0` (No event) after checking the booking status.

I did not fill other missing values because I could not safely guess the actual time, booking amount, ride distance, ratings, payment method, or reasons. Filling them with incorrect values could affect my analysis.

**Conclusion:** I replaced missing values with `0` only where I considered it appropriate and kept the remaining missing values to avoid making unsupported assumptions.

In [29]:
# Check unique booking statuses
print("Booking Status:")
print(df_clean["Booking Status"].value_counts())

# Check unique vehicle types
print("\nVehicle Type:")
print(df_clean["Vehicle Type"].value_counts())

# Check unique payment methods
print("\nPayment Method:")
print(df_clean["Payment Method"].value_counts())

Booking Status:
Booking Status
Completed                93000
Cancelled by Driver      27000
No Driver Found          10500
Cancelled by Customer    10500
Incomplete                9000
Name: count, dtype: int64

Vehicle Type:
Vehicle Type
Auto             37419
Go Mini          29806
Go Sedan         27141
Bike             22517
Premier Sedan    18111
eBike            10557
Uber XL           4449
Name: count, dtype: int64

Payment Method:
Payment Method
UPI            45909
Cash           25367
Uber Wallet    12276
Credit Card    10209
Debit Card      8239
Name: count, dtype: int64


In [30]:
# Final validation
print("Final dataset shape:", df_clean.shape)

print("Duplicate rows:", df_clean.duplicated().sum())

print("\nMissing values in each column:")
print(df_clean.isnull().sum())

print("\nFinal data types:")
print(df_clean.dtypes)

df_clean.to_csv(
    r"C:\Users\sandr\Downloads\luminar\ncr_ride_bookings_cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully!")

Final dataset shape: (150000, 21)
Duplicate rows: 0

Missing values in each column:
Date                                      0
Time                                      0
Booking ID                                0
Booking Status                            0
Customer ID                               0
Vehicle Type                              0
Pickup Location                           0
Drop Location                             0
Avg VTAT                              10500
Avg CTAT                              48000
Cancelled Rides by Customer               0
Reason for cancelling by Customer    139500
Cancelled Rides by Driver                 0
Driver Cancellation Reason           123000
Incomplete Rides                          0
Incomplete Rides Reason              141000
Booking Value                         48000
Ride Distance                         48000
Driver Ratings                        57000
Customer Rating                       57000
Payment Method                      